# 00 — Les données : température horaire à Toulouse-Blagnac

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/GalaxiLir/stats-to-deeplearning/blob/main/01-ts-uncertainty/notebooks/00_data.ipynb)

**Motivation.** Avant tout modèle, on construit la série et on la regarde : d'où viennent les données, quelles heures manquent, quel découpage chronologique garantit qu'aucune information du futur ne fuit vers l'apprentissage. Ce notebook ne contient aucun modèle ; la logique est dans `src/tsunc/data.py`.

In [ ]:
import os
import subprocess
import sys

# On Colab: clone the repository and install the package (a regular install is visible
# immediately, no session restart needed). Locally: run from the project folder.
if "google.colab" in sys.modules:
    repo = "/content/stats-to-deeplearning"
    if not os.path.exists(repo):
        url = "https://github.com/GalaxiLir/stats-to-deeplearning.git"
        subprocess.run(["git", "clone", "-q", url, repo], check=True)
    os.chdir(f"{repo}/01-ts-uncertainty")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "."], check=True)
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
print(os.getcwd())

## 1. Choisir les fichiers

Les données viennent de data.gouv.fr (Météo-France, Licence Ouverte 2.0) : un fichier par département et par période d'environ dix ans. `list_files` demande au site la liste des fichiers du département ; `select_files` garde ceux qui recouvrent la période voulue.

Pour avoir **la plus longue période possible**, mettre `START, END = None, None`.

In [ ]:
from pathlib import Path

from tsunc import data

START, END = "2010", "2023"  # inclusive bounds, UTC; None = as far as the files go
VARIABLES = ["T"]  # e.g. ["T", "TD", "U"]: codes of the columns of the source files

files = data.list_files(data.DEPARTMENT)
# If the line above fails (site unreachable, unexpected answer), build the table by hand:
# files = data.files_from_urls({"H_31_2010-2019.csv.gz": "https://...", ...})
chosen = data.select_files(files, START, END)
chosen

## 2. Télécharger et vérifier la station

Les fichiers sont gardés dans `data/raw/` (dossier ignoré par Git) : un deuxième appel ne retélécharge rien. Chaque fichier contient toutes les stations du département. Le tableau montre le nom, la position et le nombre de valeurs de chaque station dont le nom ressemble à Blagnac ou Toulouse : **si `STATION_ID` n'est pas la bonne station, on le corrige avant d'aller plus loin.**

In [ ]:
paths = data.download_files(chosen, Path("data/raw"))
for path in paths:
    print(path.name, f"{path.stat().st_size / 1e6:.1f} MB")

stations = data.list_stations(paths[-1], variable=VARIABLES[0], name_contains="BLAGNAC|TOULOUSE")
print(f"STATION_ID = {data.STATION_ID}")
stations

## 3. Construire la série

On concatène les fichiers et on place la série sur une grille horaire régulière en UTC : une heure absente devient `NaN` (on ne l'invente pas). Si deux fichiers se chevauchent, la ligne du fichier le plus récent est gardée. Le tableau de trous donne, pour chaque variable, le premier et le dernier relevé valides, le nombre d'heures manquantes et le plus long trou.

In [ ]:
frame = data.build_frame(paths, data.STATION_ID, VARIABLES, START, END)
series = frame[VARIABLES[0]]
data.gap_report(frame)

## 4. Regarder la série

Deux cycles dominent : le cycle jour-nuit (période de 24 h) et le cycle annuel. C'est pourquoi la baseline naïve saisonnière reprend la température de la veille à la même heure (m = 24). En R, `ts(x, frequency = 24)` puis `stl()` ferait apparaître les mêmes composantes.

In [ ]:
import matplotlib.pyplot as plt

year = int(series.index.year[len(series) // 2])
week = series[f"{year}-07-01":f"{year}-07-07 23:00"]
by_hour = series.groupby(series.index.hour).mean()
by_month = series.groupby(series.index.month).mean()

fig, axes = plt.subplots(1, 3, figsize=(15, 3.5))
axes[0].plot(week.index, week.to_numpy())
axes[0].set_title(f"One week of July {year}")
axes[0].tick_params(axis="x", rotation=30)
axes[1].plot(by_hour.index, by_hour.to_numpy(), marker="o")
axes[1].set_title("Mean by hour of the day (UTC)")
axes[2].plot(by_month.index, by_month.to_numpy(), marker="o")
axes[2].set_title("Mean by month")
for ax in axes:
    ax.set_ylabel("°C")
fig.tight_layout()
plt.show()

## 5. Découper en quatre blocs chronologiques

- `train` : apprentissage ; `val` : arrêt précoce et choix des hyperparamètres ; `calibration` : scores de la prédiction conforme ; `test` : évaluation finale.
- Les blocs sont disjoints et rangés dans l'ordre du temps. Pas de tirage aléatoire : la température est fortement autocorrélée, un mélange ferait fuiter le futur vers l'apprentissage.
- Les bornes sont dans `tsunc.data.BLOCK_STARTS`.

In [ ]:
import pandas as pd

blocks = data.make_blocks(series)
summary = pd.DataFrame(
    {
        name: {
            "start": block.index[0],
            "end": block.index[-1],
            "hours": len(block),
            "missing": int(block.isna().sum()),
        }
        for name, block in blocks.items()
    }
).T
print(summary)

print("Mean temperature by year (°C):")
used = pd.concat(blocks.values())
print(used.groupby(used.index.year).mean().round(2).to_string())

fig, ax = plt.subplots(figsize=(15, 3))
for name, block in blocks.items():
    daily = block.resample("D").mean()
    ax.plot(daily.index, daily.to_numpy(), linewidth=0.6, label=name)
ax.set_ylabel("°C (daily mean)")
ax.legend(ncol=4)
fig.tight_layout()
plt.show()

## Limites

- Les heures manquantes restent `NaN` : aucune imputation à ce stade ; les fenêtres contenant un trou seront écartées au moment de l'évaluation.
- Les codes qualité de Météo-France sont conservés mais pas utilisés.
- Une seule station : les résultats ne se généralisent pas telles quelles à d'autres lieux.
- Le climat évolue : les moyennes annuelles (section 5) montrent s'il y a une dérive entre l'apprentissage et le test. Une dérive viole l'échangeabilité supposée par la prédiction conforme (voir le notebook 03).
- Les fichiers sont revus régulièrement par Météo-France : noter la date de téléchargement.